# AIFM Infrastructure Fund


In this notebook we evaluate risk for a 15-year EUR 1.2bn fund
vintage 2019, with 8 assets:
* core / core-plus infrastructure
* long-duration real assets across 
    - utilities
    - energy transition
    - transport
    - social infrastructure

Assets are typically
regulated concessions or contracted, with inflation-linked cash flows and long investment horizons.

Other common infrastructure strategies for reference:
- Core-plus: similar asset types with higher leverage or development exposure
- Value-add: assets requiring operational improvement or repositioning
- Infrastructure debt: senior or subordinated lending to infra assets
- Greenfield / project finance: construction-phase assets, higher risk profile

Regulatory framework:
- AIFMD: Directive 2011/61/EU
- AIFMD II: Directive 2024/927/EU
- Delegated Regulation: EU 231/2013
- Annex IV reporting: EU 231/2013, ESMA technical guidance v1.7 (July 2024)
- Annex VI stress testing: ESMA/2020/1498
- Luxembourg implementation: Law of 12 July 2013 on AIFMs (AIFM Law)
- IPEV Valuation Guidelines (International Private Equity Valuation)

Dual UCITS/AIFM ManCo:
- CSSF Regulation 10-04 (organisational and prudential requirements)
- CSSF Regulation 22-05 (sustainability requirements, amending 10-04)

#### In this notebook

AIFM Infrastructure Fund I. Core infrastructure strategy, EUR-denominated,
long-term hold, assets held via regulated concessions and contracted structures.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.plot_style import (
    C,  FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
    apply_ax_style, section_title, fig_header, fig_footer,
    callout_box, threshold_vline, threshold_hline, breach_fill,
    pct_color, rag_color, util_color, liq_color, table_cell, sup_title,
)
from src.nb_utils import save_fig, save_table, styled_table, save_table_png

from src.database import (
    get_engine,
    InfraFund, InfraAsset, InfraFundInvestment,
    InfraCashFlow, InfraNavHistory, InfraValuationReport,
    InfraDebt, InfraCovenant,
)
from src.infra_utils import (
    fund_nav_timeseries, asset_nav_breakdown, infra_multiples,
    infra_irr, dscr_profile, ltv_profile, concentration_by_sector,
    cashflow_coverage, inflation_sensitivity, duration_profile,
    stress_nav,
)
from sqlalchemy.orm import Session
import sqlalchemy as sa

FUND_ID        = 'AIFM_Infra_Core'
VALUATION_DATE = '2026-05-13'
QUARTER        = '2026-03-31'   # latest appraiser quarter
ENGINE         = get_engine()


## 0. Setup and Validation

Infrastructure fund data is stored in eight dedicated SQLite tables, populated by
`generate_infra_fund.py` and queried here via `infra_utils.py`. The data flow mirrors
the PE fund: a synthetic generator stands in for the fund administrator system
(eFront, Allvue) that would provide this data in production.

> **Governance note**: Valuation inputs are treated as external, consistent with AIFM
> operational practice where risk management is independent from portfolio management
> and valuation.


In [ ]:
infra_tables = [
    'infra_funds', 'infra_assets', 'infra_fund_investments',
    'infra_cash_flows', 'infra_nav_history', 'infra_valuation_report',
    'infra_debt', 'infra_covenants',
]

existing = sa.inspect(ENGINE).get_table_names()

print('Table check:')
for tbl in infra_tables:
    status = '✓' if tbl in existing else '✗ MISSING'
    print(f'  {status}  {tbl}')

with ENGINE.connect() as conn:
    n_assets  = conn.execute(sa.text('SELECT COUNT(*) FROM infra_assets')).scalar()
    n_vr      = conn.execute(sa.text('SELECT COUNT(*) FROM infra_valuation_report')).scalar()
    n_cov     = conn.execute(sa.text('SELECT COUNT(*) FROM infra_covenants')).scalar()
    n_breaches = conn.execute(sa.text(
        'SELECT COUNT(*) FROM infra_covenants WHERE dscr_breach=1 OR ltv_breach=1'
    )).scalar()
    covered   = conn.execute(sa.text(
        'SELECT COUNT(DISTINCT asset_id) FROM infra_valuation_report WHERE date = :q'
    ), {'q': QUARTER}).scalar()

print(f'\nAssets loaded        : {n_assets} / 8')
print(f'Valuation reports    : {n_vr}')
print(f'Covenant readings    : {n_cov}')
print(f'Covenant breaches    : {n_breaches}')
print(f'Coverage {QUARTER} : {covered} / 8 assets  {"✓" if covered == 8 else "⚠ INCOMPLETE"}')


## 1. Portfolio Overview

An infrastructure fund does not hold daily-priced positions. Data is organised around
the asset portfolio: asset master data from the fund administrator, quarterly valuations
from independent appraisers, and cash flows between the fund and its investors.

Tables used: `infra_funds`, `infra_assets`, `infra_fund_investments`, `infra_valuation_report`.


In [ ]:
with Session(ENGINE) as session:
    fund        = session.query(InfraFund).filter_by(fund_id=FUND_ID).first()
    assets      = session.query(InfraAsset).all()
    investments = session.query(InfraFundInvestment).filter_by(fund_id=FUND_ID).all()

# ── fund metadata ────────────────────────────────────────────────────────────
meta = pd.DataFrame([
    ('Fund name',            fund.fund_name),
    ('Strategy',             'Core / core-plus infrastructure'),
    ('Vintage year',         fund.vintage_year),
    ('Fund life',            f'{fund.fund_life_years} years'),
    ('Target size',          f'EUR {fund.target_size_eur/1e6:,.0f}M'),
    ('Committed capital',    f'EUR {fund.committed_eur/1e6:,.0f}M'),
    ('Drawn capital',        f'EUR {fund.drawn_eur/1e6:,.0f}M  ({fund.drawn_eur/fund.committed_eur*100:.1f}% of committed)'),
    ('Currency',             fund.currency),
    ('Domicile',             fund.domicile),
    ('AIFMD classification', fund.aifmd_classification),
    ('Valuation date',       VALUATION_DATE),
], columns=['', ''])
display(meta.style.set_caption('Table 1.1 — Fund Metadata').hide(axis='index'))

# ── asset breakdown ──────────────────────────────────────────────────────────
with ENGINE.connect() as conn:
    vr = pd.read_sql(
        f"SELECT asset_id, implied_equity_eur FROM infra_valuation_report "
        f"WHERE fund_id='{FUND_ID}' AND date='{QUARTER}'",
        conn
    )

asset_df = pd.DataFrame([{
    'asset_id'      : a.asset_id,
    'Asset'         : a.asset_name,
    'Sector'        : a.sector,
    'Sub-type'      : a.sub_type,
    'Country'       : a.country,
    'Concession end': pd.to_datetime(a.concession_end).year,
} for a in assets])

inv_df = pd.DataFrame([{
    'asset_id'    : i.asset_id,
    'Ownership %' : i.ownership_pct,
    'Drawn equity': i.drawn_equity,
} for i in investments])

portfolio = asset_df.merge(inv_df, on='asset_id').merge(vr, on='asset_id')
portfolio['% NAV'] = portfolio['implied_equity_eur'] / portfolio['implied_equity_eur'].sum() * 100

total_nav    = portfolio['implied_equity_eur'].sum()
total_drawn  = portfolio['Drawn equity'].sum()

display_df = portfolio.drop(columns='asset_id').rename(
    columns={'implied_equity_eur': 'Appraised equity (EUR)'}
).set_index('Asset')

display_df['Ownership %']          = display_df['Ownership %'].map('{:.1f}%'.format)
display_df['Drawn equity']         = display_df['Drawn equity'].map('EUR {:,.0f}'.format)
display_df['Appraised equity (EUR)'] = display_df['Appraised equity (EUR)'].map('EUR {:,.0f}'.format)
display_df['% NAV']                = display_df['% NAV'].map('{:.1f}%'.format)

display(display_df.style.set_caption(f'Table 1.2 — Asset Breakdown  ({QUARTER})'))

print(f'\nTotal fund NAV       : EUR {total_nav/1e6:,.1f}M')
print(f'Total drawn equity   : EUR {total_drawn/1e6:,.1f}M')
print(f'Unrealised gain      : EUR {(total_nav - total_drawn)/1e6:,.1f}M  '
      f'({(total_nav/total_drawn - 1)*100:.1f}% above cost)')


## 2. Valuation and NAV

Infrastructure assets are valued quarterly by independent appraisers using yield
capitalisation: EV = EBITDA / $r$, where $r$ is the appraiser's discount rate.
* regulated assets the discount rate reflects the **allowed WACC** set by the sector regulator; 
* contracted assets it reflects counterparty credit quality and
remaining contract life.

Risk management monitors three things here:

1. **NAV trajectory** — fund-level NAV timeseries from inception
2. **Asset contribution** — waterfall of asset equity values at the current quarter
3. **Discount rate movement** — quarter-on-quarter changes in appraiser assumptions

> A shift of more than 50bps in any asset's discount rate is flagged for risk
> committee review. Under AIFMD Article 19, risk management does not challenge
> appraiser assumptions — it documents movements and escalates material changes.


In [ ]:
nav_ts = fund_nav_timeseries(ENGINE, FUND_ID)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(nav_ts['date'], nav_ts['nav_eur'] / 1e6,
        color=ACCENT, linewidth=2, marker='o', markersize=4)
ax.fill_between(nav_ts['date'], nav_ts['nav_eur'] / 1e6, alpha=0.12, color=ACCENT)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}M'))
ax.set_ylabel('EUR')
section_title(ax, f'Fund NAV — Quarterly Timeseries', fontsize=18)
ax.grid(axis='y', linestyle=':', alpha=0.4)
plt.tight_layout()
save_fig(fig, FUND_ID, "01. NAV evolution - INFRA")
plt.show()

print(f"Inception NAV : EUR {nav_ts['nav_eur'].iloc[0]/1e6:,.1f}M  ({nav_ts['date'].iloc[0].date()})")
print(f"Current NAV   : EUR {nav_ts['nav_eur'].iloc[-1]/1e6:,.1f}M  ({nav_ts['date'].iloc[-1].date()})")
print(f"NAV growth    : {(nav_ts['nav_eur'].iloc[-1] / nav_ts['nav_eur'].iloc[0] - 1)*100:.1f}%")


In [ ]:
# ── NAV waterfall by asset ───────────────────────────────────────────────────
nav_bkdn = asset_nav_breakdown(ENGINE, FUND_ID, QUARTER)

sector_colors = {'Utilities': C['blue2'], 'Transport': C['cyan'],
                 'Energy': C['green'], 'Social': C['purple']}
colors = [sector_colors.get(s, C['dim']) for s in nav_bkdn['sector']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(nav_bkdn['asset_name'], nav_bkdn['nav_eur'] / 1e6,
               color=colors, alpha=0.85, height=0.6)
for bar, pct in zip(bars, nav_bkdn['nav_pct']):
    ax.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
            f'{pct:.1f}%', va='center', fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}M'))
section_title(ax, F'NAV by Asset — {QUARTER}', fontsize=18)
ax.grid(axis='x', linestyle=':', alpha=0.4)

from matplotlib.patches import Patch
legend = [Patch(color=c, label=s) for s, c in sector_colors.items()]
ax.legend(handles=legend, fontsize=8, loc='upper right')
plt.tight_layout()
save_fig(fig, FUND_ID, '02. NAV breakdown - INFRA')
plt.show()

# ── discount rate movement QoQ ───────────────────────────────────────────────
PREV_QUARTER = '2025-12-31'

with ENGINE.connect() as conn:
    dr_df = pd.read_sql(f"""
        SELECT vr.asset_id, a.asset_name, vr.date,
               vr.discount_rate, vr.inflation_assumption
        FROM infra_valuation_report vr
        JOIN infra_assets a ON a.asset_id = vr.asset_id
        WHERE vr.fund_id = '{FUND_ID}'
          AND vr.date IN ('{PREV_QUARTER}', '{QUARTER}')
        ORDER BY vr.asset_id, vr.date
    """, conn)

dr_pivot  = dr_df.pivot(index=['asset_id', 'asset_name'], columns='date',
                         values='discount_rate')
dr_pivot.columns.name = None
dr_pivot  = dr_pivot.rename(columns={PREV_QUARTER: 'dr_prev', QUARTER: 'dr_curr'})

inf_curr  = dr_df[dr_df['date'] == QUARTER].set_index('asset_id')['inflation_assumption']
dr_pivot['inf_curr']      = inf_curr
dr_pivot['dr_chg_bps']    = (dr_pivot['dr_curr'] - dr_pivot['dr_prev']) * 10_000
dr_pivot['flagged']       = dr_pivot['dr_chg_bps'].abs() > 50
dr_pivot = dr_pivot.reset_index()

display_dr = dr_pivot[['asset_name', 'dr_prev', 'dr_curr',
                        'dr_chg_bps', 'inf_curr', 'flagged']].copy()
display_dr.columns = ['Asset', f'DR {PREV_QUARTER}', f'DR {QUARTER}',
                       'Chg (bps)', f'Infl. assumption', 'Flagged']
display_dr[f'DR {PREV_QUARTER}'] = display_dr[f'DR {PREV_QUARTER}'].map('{:.2%}'.format)
display_dr[f'DR {QUARTER}']      = display_dr[f'DR {QUARTER}'].map('{:.2%}'.format)
display_dr['Chg (bps)']          = display_dr['Chg (bps)'].map('{:+.0f} bps'.format)
display_dr['Infl. assumption']   = display_dr['Infl. assumption'].map('{:.2%}'.format)
display_dr['Flagged']            = display_dr['Flagged'].map(lambda x: '⚠ YES' if x else '—')

display(display_dr.set_index('Asset').style.set_caption(
    'Table 2.1 — Appraiser Discount Rate Assumptions (QoQ movement)'))

n_flagged = dr_pivot['flagged'].sum()
print(f'\n{"⚠  " + str(n_flagged) + " asset(s) flagged — discount rate moved > 50bps." if n_flagged else "✓  No discount rate movements exceed the 50bps threshold."}')


## 3. Performance Metrics

Infrastructure return metrics mirror PE convention: MOIC, DPI, and RVPI computed
from cash flows and current NAV. IRR uses XIRR on actual capital call and
distribution dates.

The benchmark for a core infrastructure strategy is typically **CPI + 400bps**
net of fees — reflecting the illiquidity premium over investment-grade bonds
and the inflation-linkage characteristic of the asset class. At 2.0% CPI this
implies a **6.0% net IRR target**.

Core infrastructure IRRs sit below PE buyout (typically 12–18%) by design:
lower leverage, regulated revenues, and availability-based payments reduce
volatility but compress returns. The trade-off is a more predictable, bond-like
income stream with long duration.

**DPI** measures what has actually been returned to investors — realised value.
**RVPI** measures what remains in the portfolio at appraised value — unrealised.
Early in fund life DPI is low; it builds as assets distribute or are refinanced.

**Key fund performance metrics**

<small>

- **MOIC** (Multiple on Invested Capital): total value divided by total invested capital. A MOIC of 2.0x means every EUR 1 invested is worth EUR 2 today. Includes both realised and unrealised value.

- **DPI** (Distributed to Paid-In): cash actually returned to investors divided by capital called. DPI of 0.5x means investors have received back 50 cents for every EUR 1 called. This is the "real money" metric -- cash in hand, not paper gains.

- **RVPI** (Residual Value to Paid-In): current unrealised portfolio value divided by capital called. What is still in the ground, at current valuation. RVPI + DPI = TVPI.

- **TVPI** (Total Value to Paid-In): total value (distributed + residual) divided by paid-in capital. Equivalent to MOIC when no recycling. The headline performance number before fees.

Note: for a core infrastructure fund still within its investment period, high RVPI and low DPI is expected -- assets are long-duration and capital is not returned early. DPI only builds meaningfully post-harvest or from asset sales and refinancings.


In [ ]:
multiples = infra_multiples(ENGINE, FUND_ID)
irr       = infra_irr(ENGINE, FUND_ID)

CPI_ASSUMPTION = 0.020        # fund's own inflation assumption
TARGET_IRR     = CPI_ASSUMPTION + 0.040   # CPI + 400bps
irr_vs_target  = irr - TARGET_IRR
irr_flag       = '✓ above target' if irr >= TARGET_IRR else '⚠ below target'

# ── multiples table ──────────────────────────────────────────────────────────
mult_df = pd.DataFrame([
    ('Drawn capital',   f"EUR {multiples['drawn_capital']/1e6:,.1f}M",  ''),
    ('Distributions',   f"EUR {multiples['distributions']/1e6:,.1f}M",  ''),
    ('Current NAV',     f"EUR {multiples['nav']/1e6:,.1f}M",            ''),
    ('',                '',                                               ''),
    ('DPI',             f"{multiples['dpi']:.2f}x",                      'Realised return'),
    ('RVPI',            f"{multiples['rvpi']:.2f}x",                     'Unrealised return'),
    ('MOIC',            f"{multiples['moic']:.2f}x",                     'DPI + RVPI'),
], columns=['Metric', 'Value', 'Note'])

display(mult_df.style.set_caption('Table 3.1 — Fund Multiples').hide(axis='index'))

# ── IRR vs benchmark ─────────────────────────────────────────────────────────
irr_df = pd.DataFrame([
    ('Fund IRR (net, XIRR)',       f'{irr:.2%}',         ''),
    ('Benchmark (CPI + 400bps)',   f'{TARGET_IRR:.2%}',  f'CPI {CPI_ASSUMPTION:.1%} + 400bps'),
    ('IRR vs benchmark',           f'{irr_vs_target:+.2%}', irr_flag),
], columns=['', 'Value', 'Note'])

display(irr_df.style.set_caption('Table 3.2 — IRR vs Benchmark').hide(axis='index'))

# ── visual: MOIC decomposition ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, 2))
components = {'DPI\n(realised)': multiples['dpi'],
              'RVPI\n(unrealised)': multiples['rvpi']}
bars = ax.bar(components.keys(), components.values(),
              color=[ACCENT2, ACCENT], alpha=0.85, width=0.4)
ax.axhline(1.0, color='white', linewidth=1, linestyle='--', alpha=0.6, label='Cost (1.0x)')
for bar, val in zip(bars, components.values()):
    ax.text(bar.get_x() + bar.get_width() / 2,  val+.2,
            f'{val:.2f}x', ha='center', va='center',
            fontsize=8, color='white', fontweight='bold')
ax.set_ylabel('Multiple (x)')
section_title(ax, f'MOIC Decomposition', fontsize=9)
ax.set_ylim(0, max(multiples['moic'] * 1.15, 1.3))
ax.grid(axis='y', linestyle=':', alpha=0.3)
plt.tight_layout()
save_fig(fig, FUND_ID, '03. MOIC decomposition - INFRA')
plt.show()


## 4. Leverage and Covenant Monitoring

Infrastructure assets are typically financed with project-level debt (non-recourse
to the fund). Lenders impose financial maintenance covenants tested quarterly:

$DSCR = \frac{EBITDA_{annual}}{Principal_{annual} + Interest_{annual}} \geq \text{covenant floor}$

$LTV = \frac{Net\ Debt}{Appraised\ EV} \leq \text{covenant ceiling}$

A DSCR breach signals that operating cash flows are insufficient to service debt —
the most critical early warning in infrastructure lending. An LTV breach signals
that asset value has deteriorated relative to leverage. Typical triggers:
* construction overruns
* traffic underperformance
* sharp discount rate increase

On breach, the lender's remedy is typically a **waiver** with conditions: 

- **Cash sweep**: excess cash flow is automatically captured by lenders to repay debt ahead of schedule instead of being distributed to equity.
- **Capital injection**: sponsor puts in fresh equity to restore the financial ratio that triggered the breach.
- **Equity cure**: a specific contractual right allowing the sponsor to inject capital that counts directly toward curing the covenant calculation, within a defined cure period.
- **Enforcement (severe cases)**: lender exercises security rights -- typically step-in, asset sale, or acceleration of debt -- when breach is uncured and material.

AIFM documents breaches and waivers in the quarterly risk report and notifies the board and LPAC.

> **10% headroom flag**: any asset where current DSCR is within 10% of its
> covenant floor is flagged for enhanced monitoring.


In [ ]:
import matplotlib.dates as mdates

ASSET_IDS      = [a.asset_id for a in assets]
asset_name_map = {a.asset_id: a.asset_name for a in assets}

import io
import base64
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from IPython.display import HTML, display


def sparkline(values, breaches=None):
    fig, ax = plt.subplots(figsize=(1.8, 0.4))
    x = range(len(values))
    ax.plot(x, values, color=ACCENT, lw=1.2)
    ax.scatter(x, values, color=ACCENT, s=12, zorder=5)
    if breaches is not None:
        for i, is_breach in enumerate(breaches):
            if is_breach:
                ax.scatter(i, values[i], color=C['red'], s=20, zorder=6)
    ax.axis('off')
    fig.patch.set_alpha(0)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=80, bbox_inches='tight',
                facecolor='none', transparent=True)
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()


def covenant_table_html(engine, asset_ids, asset_name_map,
                        profile_fn, metric_col, covenant_col,
                        breach_col, breach_direction, label):

    rows = []

    for asset_id in asset_ids:
        df           = profile_fn(engine, asset_id).tail(12).sort_values('date')
        cov          = df[covenant_col].iloc[-1]
        latest       = df[metric_col].iloc[-1]
        headroom     = latest - cov if breach_direction == 'below' else cov - latest
        headroom_pct = headroom / cov
        trend        = df['trend'].iloc[-1]
        breach       = bool(df[breach_col].iloc[-1])
        waiver       = df['waiver_granted'].any()

        if breach:
            status = '🔴 Breach'
            row_bg = '#3d0000'
        elif headroom_pct < 0.10:
            status = '🟡 Watch'
            row_bg = '#2a2000'
        else:
            status = '🟢 OK'
            row_bg = '#0d1b2a'

        trend_arrow = '↑' if trend == 'improving' else ('↓' if trend == 'deteriorating' else '→')
        waiver_flag = '  ⚠️' if waiver else ''

        spark     = sparkline(df[metric_col].values, breaches=df[breach_col].values)
        spark_img = (f'<img src="data:image/png;base64,{spark}" '
                     f'style="height:32px;vertical-align:middle;">')



        breach_count = int(df[breach_col].sum())
        
        rows.append({
            'row_bg'      : row_bg,
            'Asset'       : asset_name_map[asset_id],
            label         : f'{latest:.2f}x',
            'Covenant'    : f'{cov:.2f}',
            'Last 8Q'     : spark_img,
            'Headroom'    : f'{headroom:+.2f}',
            'Hdroom %'    : f'{headroom_pct:+.1%}',
            'Breaches'    : str(breach_count) if breach_count > 0 else '—',
            'Trend'       : trend_arrow + waiver_flag,
            'Status'      : status,
        })

    cols   = ['Asset', label, 'Covenant', 'Last 8Q', 'Headroom', 'Hdroom %', 'Breaches', 'Trend', 'Status']
    header = ''.join(f'<th>{c}</th>' for c in cols)
    body   = ''
    for r in rows:
        cells  = ''.join(f'<td>{r[c]}</td>' for c in cols)
        body  += f'<tr style="background-color:{r["row_bg"]};color:white;">{cells}</tr>'

    html = f"""
    <style>
        table.cov {{ border-collapse: collapse; font-size: 11px; width: 100%; }}
        table.cov th {{ background-color: #0a0f1e; color: white; padding: 6px 12px;
                        text-align: center; border-bottom: 1px solid #333; }}
        table.cov td {{ padding: 5px 12px; text-align: center;
                        border-bottom: 1px solid #1a1a2e; }}
    </style>
    <table class="cov">
        <thead><tr>{header}</tr></thead>
        <tbody>{body}</tbody>
    </table>
    """

    return html


# ── DSCR ──────────────────────────────────────────────────────────────────────
html = covenant_table_html(
    ENGINE, ASSET_IDS, asset_name_map,
    profile_fn       = dscr_profile,
    metric_col       = 'dscr_actual',
    covenant_col     = 'dscr_covenant',
    breach_col       = 'dscr_breach',
    breach_direction = 'below',
    label            = 'DSCR',
)

display(HTML('<h4 style="color:white;">DSCR Covenant Monitor</h4>' + html))
save_table_png(html, FUND_ID, '04. DSRC timeseries and headroom - INFRA')

# ── LTV ───────────────────────────────────────────────────────────────────────
html = covenant_table_html(
    ENGINE, ASSET_IDS, asset_name_map,
    profile_fn      = ltv_profile,
    metric_col      = 'ltv_actual',
    covenant_col    = 'ltv_covenant',
    breach_col      = 'ltv_breach',
    breach_direction= 'above',
    label           = 'LTV',
)
display(HTML(f'<h4 style="color:white;">LTV Covenant Monitor</h4>' + html))
save_table_png(html, FUND_ID, '05. LTV timeseries and headroom - INFRA')




## 5. Concentration Risk

Infrastructure funds face concentration risk primarily along three dimensions:
**sector**, **country**, and **asset sub-type**. Unlike liquid funds where
concentration limits are set by UCITS rules or fund prospectus, AIFM infrastructure
funds set internal policy limits approved by the risk committee.

The fund's internal risk policy sets a **40% NAV sector concentration limit**.
This is a common threshold for core infrastructure strategies: a single sector
above 40% implies over-reliance on one regulatory regime or demand driver.

**Sub-type** is a qualitative dimension:
- **Regulated**: revenues set by a sector regulator (e.g. Bundesnetzagentur,
  CRE) — lowest risk, most predictable
- **Contracted**: fixed-price offtake or availability payments — counterparty
  credit risk replaces regulatory reset risk
- **Concession**: volume risk (e.g. toll roads, airports) — more cyclical,
  more correlated with economic activity

> AIFMD does not prescribe sector concentration limits for closed-end funds.
> The 40% threshold is this fund's own internal policy. It would be disclosed
> in the prospectus and reported in the Annex IV filing.


In [ ]:
conc = concentration_by_sector(ENGINE, FUND_ID, QUARTER)

# ── table ────────────────────────────────────────────────────────────────────
conc_display = conc.copy()
conc_display['nav_eur'] = conc_display['nav_eur'].map('EUR {:,.0f}'.format)
conc_display['nav_pct'] = conc_display['nav_pct'].map('{:.1f}%'.format)
conc_display['concentrated'] = conc_display['concentrated'].map(
    lambda x: '⚠ BREACH' if x else '—')
conc_display.columns = ['Sector', 'NAV (EUR)', '% NAV', 'Concentration flag']
display(conc_display.set_index('Sector').style.set_caption(
    f'Table 5.1 — Sector Concentration ({QUARTER})  |  Limit: 40% NAV'))

n_breaches = conc['concentrated'].sum()
print(f'\n{"⚠  " + str(n_breaches) + " sector(s) exceed the 40% NAV concentration limit." if n_breaches else "✓  No sector breaches the 40% NAV concentration limit."}')



In [ ]:
with ENGINE.connect() as conn:
    detail = pd.read_sql(f"""
        SELECT a.country, a.sub_type,
               vr.implied_equity_eur
        FROM infra_valuation_report vr
        JOIN infra_assets a ON a.asset_id = vr.asset_id
        WHERE vr.fund_id = '{FUND_ID}' AND vr.date = '{QUARTER}'
    """, conn)

total_nav = detail['implied_equity_eur'].sum()

country = (detail.groupby('country')['implied_equity_eur'].sum()
           .sort_values(ascending=False)
           .reset_index())
country['nav_pct'] = country['implied_equity_eur'] / total_nav * 100

subtype = (detail.groupby('sub_type')['implied_equity_eur'].sum()
           .sort_values(ascending=False)
           .reset_index())
subtype['nav_pct'] = subtype['implied_equity_eur'] / total_nav * 100

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

# ── country ──────────────────────────────────────────────────────────────────
bars1 = ax1.bar(country['country'], country['nav_pct'],
                color=ACCENT, alpha=0.85, width=0.5)
for bar, pct in zip(bars1, country['nav_pct']):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.5, f'{pct:.1f}%',
             ha='center', fontsize=9, fontweight='bold')
ax1.set_ylabel('% NAV')
section_title(ax1, 'Country Concentration', fontsize=10)

# ── sub-type ─────────────────────────────────────────────────────────────────
bars2 = ax2.barh(subtype['sub_type'], subtype['nav_pct'],
                 color=ACCENT, alpha=0.85, height=0.5)
for bar, pct in zip(bars2, subtype['nav_pct']):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{pct:.1f}%', va='center', fontsize=9, fontweight='bold')
ax2.set_xlabel('% NAV')
section_title(ax2, 'Sub-type Mix', fontsize=10)

# ── sector ───────────────────────────────────────────────────────────────────
bar_colors = [C['red'] if c else ACCENT for c in conc['concentrated']]
bars3 = ax3.barh(conc['sector'], conc['nav_pct'],
                 color=bar_colors, alpha=0.85, height=0.5)
ax3.axvline(40.0, color=C['red'], linewidth=1, linestyle='--')
ax3.text(40.5, 0.98, '40% limit', color=C['red'], fontsize=8,
         va='top', transform=ax3.get_xaxis_transform())
for bar, pct in zip(bars3, conc['nav_pct']):
    ax3.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{pct:.1f}%', va='center', fontsize=9, fontweight='bold')
ax3.set_xlabel('% NAV')
ax3.set_xlim(0, max(conc['nav_pct'].max() * 1.2, 50))
section_title(ax3, 'Sector Concentration', fontsize=10)

sup_title(fig, f'Concentration — Country, Sub-type and Sector  ({QUARTER})', fontsize=13)
plt.tight_layout()
save_fig(fig, FUND_ID, '04. Concentration by country, subtype, sector - INFRA')
plt.show()

## 6. Inflation and Duration

Two structural characteristics define infrastructure as an asset class: **inflation
linkage** and **long duration**. They drive valuation sensitivity and long-term cash flow predictability.

**Inflation linkage** measures what proportion of an asset's revenues automatically
adjust with CPI or PPI. Regulated assets (water, electricity) have tariffs reset
by the regulator with explicit CPI indexation. Availability-based PPPs typically
link fees to RPI. Toll roads and airports have partial linkage via tariff escalation
clauses but retain volume risk. Merchant assets have none.

$$\text{Weighted linkage} = \sum_{i} w_i \times \lambda_i$$

where $w_i$ is asset $i$'s share of fund NAV and $\lambda_i \in [0, 1]$ is its
inflation linkage coefficient from the appraiser's model.

**Duration** in infrastructure refers to remaining concession or contract life.
Concession expiry is a hard terminal event — unlike a bond, there is no
principal repayment. Assets approaching expiry within 3 years require an active
exit or re-tendering strategy, and must be flagged in the Annex IV filing.


In [ ]:
inf = inflation_sensitivity(ENGINE, FUND_ID)

# ── summary table ────────────────────────────────────────────────────────────
summary = pd.DataFrame([
    ('Weighted avg. inflation linkage', f"{inf['weighted_avg_linkage']:.1%}",
     'NAV-weighted average across all 8 assets'),
    ('% fully CPI-linked',              f"{inf['pct_fully_linked']:.1f}%",
     'Regulated tariffs or full CPI pass-through'),
    ('% partially linked',              f"{inf['pct_partially_linked']:.1f}%",
     'Partial indexation or price review mechanism'),
    ('% unlinked',                      f"{inf['pct_unlinked']:.1f}%",
     'Fixed-fee or merchant exposure'),
], columns=['Metric', 'Value', 'Description'])
display(summary.style.set_caption('Table 6.1 — Inflation Sensitivity Summary').hide(axis='index'))

# ── asset detail ─────────────────────────────────────────────────────────────
ad = pd.DataFrame(inf['asset_detail']).sort_values('inflation_linkage', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# per-asset linkage bar
bar_colors = [ACCENT if v >= 0.75 else ACCENT2 if v >= 0.40 else C['red']
              for v in ad['inflation_linkage']]
bars = ax1.barh(ad['asset_name'], ad['inflation_linkage'] * 100,
                color=bar_colors, alpha=0.85, height=0.55)
ax1.axvline(75, color=ACCENT, linewidth=1, linestyle='--', alpha=0.6, label='Fully linked (75%)')
ax1.axvline(40, color=ACCENT2, linewidth=1, linestyle='--', alpha=0.6, label='Partial threshold (40%)')
for bar, v in zip(bars, ad['inflation_linkage']):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
             f'{v:.0%}', va='center', fontsize=8)
ax1.set_xlabel('Inflation linkage (%)')
# ax1.set_title('Inflation Linkage by Asset', fontsize=10, fontweight='bold')
section_title(ax1, f'Inflation Linkage by Asset', fontsize=11)

ax1.set_xlim(0, 115)
ax1.legend(fontsize=7)
ax1.grid(axis='x', linestyle=':', alpha=0.3)
ax1.spines[['top', 'right']].set_visible(False)

# linkage mix donut
sizes  = [inf['pct_fully_linked'], inf['pct_partially_linked'], inf['pct_unlinked']]
labels = ['Fully linked', 'Partially linked', 'Unlinked']
clrs   = [ACCENT, ACCENT2, C['muted']]
non_zero = [(s, l, c) for s, l, c in zip(sizes, labels, clrs) if s > 0]
if non_zero:
    s, l, c = zip(*non_zero)
    wedges, texts, autotexts = ax2.pie(
        s, labels=l, colors=c, autopct='%1.1f%%',
        startangle=90, pctdistance=.7,
        wedgeprops=dict(width=0.7))
    for t in autotexts:
        t.set_fontsize(9)
# ax2.set_title('Linkage Mix (% NAV)', fontsize=10, fontweight='bold')
section_title(ax2, f'Linkage Mix (% NAV)', fontsize=11)

# plt.suptitle(f'Inflation Sensitivity  |  Weighted avg. linkage: {inf["weighted_avg_linkage"]:.1%}',
#              fontsize=11, y=1.01)

sup_title(fig, f'Inflation Sensitivity  |  Weighted avg. linkage: {inf["weighted_avg_linkage"]:.1%}', fontsize=18)

plt.tight_layout()
save_fig(fig, FUND_ID, '05. Inflation sensitivity INFRA')
plt.show()


In [ ]:
dur = duration_profile(ENGINE, FUND_ID)
weighted_avg = dur.attrs['weighted_avg_remaining_years']

# ── table ────────────────────────────────────────────────────────────────────
dur_display = dur[['asset_name', 'concession_end', 'remaining_years',
                    'nav_weight', 'near_expiry']].copy()
dur_display['concession_end']  = pd.to_datetime(dur_display['concession_end']).dt.strftime('%Y-%m-%d')
dur_display['remaining_years'] = dur_display['remaining_years'].map('{:.1f} yrs'.format)
dur_display['nav_weight']      = dur_display['nav_weight'].map('{:.1%}'.format)
dur_display['near_expiry']     = dur_display['near_expiry'].map(lambda x: '⚠ YES' if x else '—')
dur_display.columns = ['Asset', 'Concession end', 'Remaining years', 'NAV weight', 'Near expiry (<3 yr)']

display(dur_display.set_index('Asset').style.set_caption(
    f'Table 6.2 — Concession Duration Profile  |  Weighted avg: {weighted_avg:.1f} years'))

# ── chart ────────────────────────────────────────────────────────────────────
dur_sorted = dur.sort_values('remaining_years', ascending=True)
bar_colors = [C['red'] if x else ACCENT for x in dur_sorted['near_expiry']]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(dur_sorted['asset_name'], dur_sorted['remaining_years'],
               color=bar_colors, alpha=0.85, height=0.55)
ax.axvline(3, color=C['red'], linewidth=1.5, linestyle='--', alpha=0.7, label='3-year expiry flag')
ax.axvline(weighted_avg, color='white', linewidth=1.5, linestyle=':',
           alpha=0.7, label=f'Weighted avg ({weighted_avg:.1f} yrs)')
for bar, yrs in zip(bars, dur_sorted['remaining_years']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{yrs:.1f} yrs', va='center', fontsize=10)
ax.set_xlabel('Remaining concession life (years)')
section_title(ax, 'Concession Duration Profile', fontsize=15)

ax.legend(fontsize=10)
ax.grid(False)
plt.tight_layout()
save_fig(fig, FUND_ID, '06. Concession duration profile - INFRA')
plt.show()

n_near = dur['near_expiry'].sum()
print(f'\nWeighted avg remaining life : {weighted_avg:.1f} years')
print(f'Near-expiry assets (<3 yr)  : {"⚠ " + str(n_near) if n_near else "None — ✓ all assets have 3+ years remaining"}')

## 7. Cashflow and Liquidity

In a closed-end infrastructure fund the fund is designed to hold to concession maturity or a planned exit. Liquidity risk lives entirely on the
**liability side**: the fund must be able to meet capital call obligations,
management fees, and debt service at the asset level.

The shape of the J-curve depends heavily on the strategy. 
* Core infrastructure 
funds investing in operational assets generate cash flow from day one and 
distribute early. The J-curve is shallow or almost flat. Value-add and 
core-plus funds with repositioning or capex programmes will see a deeper trough. 

* Greenfield and project finance funds are similar to teh PE J curve shape. Assets consume 
capital through construction before generating any operating cash flow, and 
distributions only begin once projects reach commercial operation, which can 
take three to five years or longer.

**Cashflow coverage** measures whether quarterly distributions from assets are
sufficient to cover the fund's management fees without requiring LP capital calls.
A ratio above 1.0x means the fund is self-funding its operating costs from income.

**Unfunded commitments** are the LP capital not yet drawn. For a 2019-vintage
15-year fund at EUR 1.2bn committed, remaining unfunded represents the liquidity
obligation the ManCo must be able to call from investors at short notice. Under
AIFMD Article 16, the AIFM must ensure the fund's liquidity profile matches its
redemption policy — here: closed-end, no redemptions, unfunded drawn by capital
call notice.


In [ ]:
with ENGINE.connect() as conn:
    cf_df = pd.read_sql(f"""
        SELECT date, flow_type, SUM(amount_eur) as amount_eur
        FROM infra_cash_flows
        WHERE fund_id = '{FUND_ID}'
        GROUP BY date, flow_type
        ORDER BY date
    """, conn)

cf_df['date']    = pd.to_datetime(cf_df['date'])
cf_df['quarter'] = cf_df['date'].dt.to_period('Q').dt.to_timestamp('Q')

calls = (cf_df[cf_df['flow_type'] == 'capital_call']
         .groupby('quarter')['amount_eur'].sum().abs().rename('calls'))
fees  = (cf_df[cf_df['flow_type'] == 'management_fee']
         .groupby('quarter')['amount_eur'].sum().abs().rename('fees'))
dist  = (cf_df[cf_df['flow_type'] == 'distribution']
         .groupby('quarter')['amount_eur'].sum().rename('distributions'))

quarters = calls.index.union(fees.index).union(dist.index)
cf = pd.DataFrame(index=quarters)
cf = cf.join(calls).join(fees).join(dist).fillna(0).sort_index()

cf['ncf']  = cf['distributions'] - cf['calls'] - cf['fees']
cf['cncf'] = cf['ncf'].cumsum()

trough_idx = cf['cncf'].idxmin()
trough_val = cf.loc[trough_idx, 'cncf']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5),
                                gridspec_kw={'height_ratios': [2, 1]},
                                sharex=True)

ax1.bar(cf.index, -cf['calls'],         width=60, color=C['red'], alpha=0.75, label='Capital calls')
ax1.bar(cf.index, -cf['fees'],          width=60, color=C['amber'], alpha=0.65,
        bottom=-cf['calls'], label='Management fees')
ax1.bar(cf.index, cf['distributions'],  width=60, color=C['green'], alpha=0.75, label='Distributions')
ax1.plot(cf.index, cf['cncf'], color='white', linewidth=2.2, marker='o',
         markersize=3.5, label='Cumulative NCF')
ax1.axhline(0, color='grey', linewidth=0.8, linestyle='--')
ax1.annotate(
    f'Trough\n{trough_idx.strftime("%b %Y")}\n{trough_val/1e6:.0f}M',
    xy=(trough_idx, trough_val),
    xytext=(trough_idx, trough_val * 0.65),
    arrowprops=dict(arrowstyle='->', color=C['red']),
    fontsize=8, color=C['red'], ha='center'
)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax1.set_ylabel('EUR')
# ax1.set_title('Infrastructure J-Curve — AIFM Infra Core', fontsize=12, fontweight='bold')
section_title(ax1, 'J-Curve - INFRA fund', fontsize=14)

ax1.legend(fontsize=8, loc='lower left')
ax1.spines[['top', 'right']].set_visible(False)

cf['cum_calls'] = cf['calls'].cumsum()
cf['cum_dist']  = cf['distributions'].cumsum()
cf['dpi']  = (cf['cum_dist'] / cf['cum_calls'].replace(0, float('nan')))
cf['pic']  = cf['cum_calls']

ax2.plot(cf.index, cf['dpi'], color=ACCENT, linewidth=2, label='DPI (distributions / called)')
ax2.axhline(1.0, color='grey', linewidth=0.8, linestyle='--')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.2f}x'))
ax2.set_ylabel('DPI (x)')
ax2.legend(fontsize=8, loc='upper left')
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
save_fig(fig, FUND_ID, '07. Infrastructure J-Curve')
plt.show()


print(f"Total calls        : EUR {cf['calls'].sum()/1e6:,.1f}M")
print(f"Total distributions: EUR {cf['distributions'].sum()/1e6:,.1f}M")
print(f"Cumulative NCF     : EUR {cf['cncf'].iloc[-1]/1e6:,.1f}M")
print(f"DPI (to date)      : {cf['dpi'].iloc[-1]:.2f}x")


In [ ]:
ccov = cashflow_coverage(ENGINE, FUND_ID)

fig, ax = plt.subplots(figsize=(6, 4))
ax2_right = ax.twinx()

# stacked bars — fees on top of distributions
ax.bar(ccov['date'], ccov['distributions'] / 1e6,
       width=60, color=C['green'], alpha=0.7, label='Distributions')
ax.bar(ccov['date'], ccov['management_fees'] / 1e6,
       width=60, color=C['amber'], alpha=0.85, label='Management fees',
       bottom=ccov['distributions'] / 1e6)

# filter out zero coverage ratio points so line does not crash to 0
cov_valid = ccov[ccov['coverage_ratio'] > 0]
ax2_right.plot(cov_valid['date'], cov_valid['coverage_ratio'],
               color='white', linewidth=2, marker='o', markersize=3.5,
               label='Coverage ratio (right)')
ax2_right.axhline(1.0, color=C['red'], linewidth=1.2, linestyle='--', alpha=0.7)

ax.set_ylabel('EUR M')
ax2_right.set_ylabel('Coverage ratio (x)')
section_title(ax, '8. Cashflow Coverage — Distributions vs Management Fees', fontsize=14)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2_right.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')
ax.grid(axis='y', linestyle=':', alpha=0.3)
ax.grid(False)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))
plt.tight_layout()
save_fig(fig, FUND_ID, '08. Cashflow Coverage — Distributions vs Management Fees')
plt.show()

## 8. Stress Testing

Infrastructure NAV is driven by two model inputs: the **discount rate** used by
the appraiser and the **EBITDA**, which is itself partially a function of inflation
linkage. Stress testing here is not market risk stress — it is **valuation model
input stress**, targeting the assumptions that most influence appraised equity value.

$$EV_{stressed} = \frac{EBITDA \times (1 + \lambda \cdot \Delta\pi)}{r + \Delta r}$$

$$Equity_{stressed} = \max(0,\ EV_{stressed} - Net\ Debt)$$

where $\lambda$ is the asset's inflation linkage coefficient, $\Delta\pi$ is the
inflation shock, and $\Delta r$ is the discount rate shock in bps.

Three scenarios:

| Scenario | $\Delta r$ | $\Delta\pi$ | Rationale |
|---|---|---|---|
| (a) Rate shock | +100bps | 0 | Interest rate rise → higher WACC → lower EV |
| (b) Inflation loss | 0 | −2% | Deflation or loss of indexation → lower contracted revenues |
| (c) Combined | +100bps | −2% | Adverse macro: rate rise with falling inflation (stagflation reversal) |

> *Stress scenarios applied to appraiser discount rate and inflation assumptions
> to assess sensitivity of NAV to valuation model inputs. Consistent with
> ESMA/2020/1498 (Annex VI) stress testing guidance and AIFMD Article 16(1).*


In [ ]:
scenarios = {
    '(a) +100bps rate shock':   stress_nav(ENGINE, FUND_ID, 100,  0.00),
    '(b) −2% Inflation shock':  stress_nav(ENGINE, FUND_ID,   0, -0.02),
    '(c) Combined':             stress_nav(ENGINE, FUND_ID, 100, -0.02),
}

base_nav = list(scenarios.values())[0]['base_nav']

# ── summary table ─────────────────────────────────────────────────────────────
rows = []
for label, res in scenarios.items():
    rows.append({
        'Scenario'          : label,
        'Base NAV'          : f"EUR {res['base_nav']/1e6:,.1f}M",
        'Stressed NAV'      : f"EUR {res['stressed_nav']/1e6:,.1f}M",
        'NAV change (EUR)'  : f"EUR {res['nav_change']/1e6:,.1f}M",
        'NAV change (%)'    : f"{res['nav_change_pct']/100:+.2%}",
    })

stress_tbl = pd.DataFrame(rows).set_index('Scenario')
display(stress_tbl.style.set_caption('NAV Stress Scenarios'))

# ── bar chart ─────────────────────────────────────────────────────────────────
labels  = list(scenarios.keys())
impacts = [res['nav_change_pct'] for res in scenarios.values()]
colors  = [C['red'] if v < 0 else C['green'] for v in impacts]

fig, ax = plt.subplots(figsize=(6, 2))
bars = ax.barh(labels, impacts, color=colors, alpha=0.85, height=0.45)
ax.axvline(0, color='white', linewidth=0.8)
ax.set_xlim(min(impacts) * 1.1, 0)


for bar, val in zip(bars, impacts):
    ax.text(bar.get_width() - 0.3 if val < 0 else bar.get_width() + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.1f}%', va='center', ha='right' if val < 0 else 'left',
            fontsize=8, fontweight='bold',
            color='white' if val < 0 else 'white')
ax.set_xlabel('NAV change (%)')
section_title(ax, f'Stress Impact on Fund NAV', fontsize=12)

ax.grid(False)
ax.yaxis.set_label_position('right')
ax.yaxis.tick_right()
plt.tight_layout()
save_fig(fig, FUND_ID, '09. stress test (fund) INFRA')
plt.show()


In [ ]:
combined = scenarios['(c) Combined']

detail = pd.DataFrame(combined['asset_detail'])
detail['nav_change_pct'] = detail['nav_change'] / detail['base_nav'] * 100
detail = detail.sort_values('nav_change')

detail_display = detail[['asset_name', 'base_nav', 'stressed_nav',
                          'nav_change', 'nav_change_pct']].copy()
detail_display.columns = ['Asset', 'Base NAV', 'Stressed NAV',
                           'Change', 'Change (%)']
detail_display['Base NAV']     = detail_display['Base NAV'].map(lambda x: f'{x/1e6:,.1f}')
detail_display['Stressed NAV'] = detail_display['Stressed NAV'].map(lambda x: f'{x/1e6:,.1f}')
detail_display['Change']       = detail_display['Change'].map(lambda x: f'{x/1e6:+,.1f}')
detail_display['Change (%)']          = detail_display['Change (%)'].map(lambda x: f'{x:+.1f}%')

display(detail_display.set_index('Asset').style.set_caption(
    'Asset-Level Stress Impact (EUR mm) |  Scenario (c): DR +100bps + Inflation −2%'))

# ── asset-level bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
pcts = detail['nav_change_pct']
bars = ax.barh(detail['asset_name'], pcts,
               color=C['red'], alpha=0.80, height=0.55)
for bar, val in zip(bars, pcts):
    ax.text(bar.get_width() - 0.3 if val < 0 else bar.get_width() + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f'{val:+.2f}%', va='center', ha='right' if val < 0 else 'left',
            fontsize=10, fontweight='bold',
            color='white' if val < 0 else 'white')
ax.axvline(0, color='white', linewidth=0.8)
ax.set_xlabel('NAV change (%)')
ax.set_xlim(min(pcts) * 1.1, 0)
section_title(ax, 'Asset-Level Stress Impact  |  Scenario (c): Combined', fontsize=15)
ax.grid(False)
ax.yaxis.set_label_position('right')
ax.yaxis.tick_right()
plt.tight_layout()
save_fig(fig, FUND_ID, '10. Asset-Level Stress Impact (combined scenario)')
plt.show()

print(f'\nBase NAV       : EUR {combined["base_nav"]/1e6:,.1f}M')
print(f'Stressed NAV   : EUR {combined["stressed_nav"]/1e6:,.1f}M')
print(f'NAV at risk    : EUR {abs(combined["nav_change"])/1e6:,.1f}M  '
      f'({combined["nav_change_pct"]:+.1f}%)')
print(f'\nMost sensitive : {detail.iloc[0]["asset_name"]}  '
      f'({detail.iloc[0]["nav_change_pct"]:+.1f}%)')
print(f'Least sensitive: {detail.iloc[-1]["asset_name"]}  '
      f'({detail.iloc[-1]["nav_change_pct"]:+.1f}%)')


---

## 9. Annex IV Report — MRS-78

Closed-ended infrastructure AIFs have specific Annex IV requirements under
AIFMD Article 110 / EU 231/2013:

- **Fund structure**: closed-ended with no periodic redemption. AIFMD Article 15
  liquidity management requirements are less stringent — the focus is on matching
  fund life to the concession horizon of the underlying assets.
- **Leverage**: infrastructure funds typically have no financial leverage at fund level.
  Project-level debt (senior debt within each SPV) is ring-fenced and excluded from
  AIFMD leverage per EU 231/2013 Art. 7 project finance carve-out. Both are disclosed.
- **Key metrics**: unlike liquid funds, the report emphasises concession duration,
  inflation linkage, and DSCR/LTV covenant status rather than daily VaR.
- **AIFMD Art. 19 boundary**: NAV is set by an independent appraiser. The AIFM
  discloses appraiser identity but does not challenge or override appraisal inputs.

**Regulatory basis:** AIFMD Art. 110 / EU 231/2013 Annex IV / IPEV Valuation Guidelines


In [ ]:
from src.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"Fund NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

print("\n=== Section 1 — Fund Identification ===")
display(annex_iv['identification'])


In [ ]:
# Section 2 — Exposures (asset-level NAV from independent appraisal)
print("\n=== Section 2.1 — Asset NAV Breakdown ===")
display(annex_iv['asset_breakdown'])

print("\n=== Section 2.2 — Sector Concentration ===")
display(annex_iv['sector_breakdown'])

print("\n=== Section 2.3 — Geographic Breakdown ===")
display(annex_iv['country_breakdown'])

print("\n=== Section 2.4 — Top 5 Assets by NAV ===")
display(annex_iv['top5_positions'])


In [ ]:
# Section 3 — Leverage (fund level: none; project level: disclosed separately)
print("\n=== Section 3 — Leverage ===")
display(annex_iv['leverage_detail'])

# Section 5 — Performance and infrastructure characteristics
print("\n=== Section 5 — Fund Performance & Infrastructure Characteristics ===")
display(annex_iv['performance'])


## ESG Risk Indicators

Infrastructure assets are assessed by independent appraisers each quarter alongside the
financial valuation. `build_private_esg_df` with `asset_type='infra'` uses the appraiser
field from `infra_valuation_report` as `esg_reporter`.

Infrastructure ESG profiles vary significantly by sector. Offshore wind and social
infrastructure score well on environmental and social metrics respectively. Airports and
toll roads carry high carbon intensity — a key PAI indicator under SFDR and the expanded
Annex IV under AIFMD II. Assets with controversy flags (INFRA_003, INFRA_007) correspond
to the DSCR and LTV covenant events already identified in the covenant section of this notebook.


In [ ]:
from src.esg_utils import build_private_esg_df, esg_portfolio_summary, ESG_THRESHOLD

CURRENT_QUARTER = '2026-03-31'

esg_df  = build_private_esg_df(FUND_ID, CURRENT_QUARTER, 'infra', ENGINE)
NAV_ESG = esg_df['esg_exposure_eur'].sum()
summary = esg_portfolio_summary(esg_df, NAV_ESG)

print("INFRASTRUCTURE PORTFOLIO ESG SUMMARY")
print(f"  Weighted avg ESG score  : {summary['wav_esg']:.1f}")
print(f"  Weighted avg ENV score  : {summary['wav_env']:.1f}")
print(f"  Weighted avg SOC score  : {summary['wav_soc']:.1f}")
print(f"  Weighted avg GOV score  : {summary['wav_gov']:.1f}")
print(f"  Avg carbon intensity    : {summary['wav_carbon']:.1f} tCO2e / EUR M")
print(f"  % below ESG threshold   : {summary['pct_low_esg']:.1f}%  (threshold = {ESG_THRESHOLD})")
print(f"  % controversy exposure  : {summary['pct_controversy']:.1f}%")
print()

esg_df[['instrument_name', 'sub_asset_class', 'esg_score', 'env_score',
        'soc_score', 'gov_score', 'controversy_flag',
        'carbon_intensity', 'esg_reporter', 'weight_pct']].style \
    .background_gradient(subset=['esg_score'], cmap='RdYlGn', vmin=0, vmax=100) \
    .format({'esg_score': '{:.0f}', 'env_score': '{:.0f}', 'soc_score': '{:.0f}',
             'gov_score': '{:.0f}', 'carbon_intensity': '{:.1f}', 'weight_pct': '{:.1f}%'})
